<a href="https://colab.research.google.com/github/selim679/databricks-streaming-pipeline/blob/main/01_ingestion_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

checkpoint_path = "/Volumes/workspace/default/youtubeseries/checkpoints/bronze"
bronze_path = "/Volumes/workspace/default/youtubeseries/delta/bronze"

schema = StructType([
    StructField("user_id", IntegerType()),
    StructField("event_type", StringType()),
    StructField("device", StringType()),
    StructField("amount", DoubleType()),
    StructField("timestamp", TimestampType())
])

df_stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

df_events = df_stream.select(
    (col("value") % 100).alias("user_id"),
    when((col("value") % 3) == 0, "click")
    .when((col("value") % 3) == 1, "view")
    .otherwise("purchase").alias("event_type"),
    when((col("value") % 2) == 0, "mobile")
    .otherwise("desktop").alias("device"),
    (rand() * 100).alias("amount"),
    col("timestamp")
)

df_events.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .start(bronze_path)

In [ ]:
df = spark.read.format("delta").load(bronze_path)
display(df)
